# Notebook 2: Feature Engineering

This notebook transforms the raw OULAD tables into a machine learning-ready dataset for the SEAID Framework.

Feature engineering includes:

- Student engagement metrics
- Assessment performance metrics
- Registration behavior
- Demographic variables
- Encoded categorical variables
- Target variable creation
- Missing value handling

The resulting dataset serves as the standardized input for all predictive models developed in subsequent notebooks.

In [1]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

In [3]:
PROJECT_ROOT = Path("/content/drive/MyDrive/SEAID_Framework")

print(PROJECT_ROOT)
print(PROJECT_ROOT.exists())

/content/drive/MyDrive/SEAID_Framework
True


In [4]:
import os

for item in sorted(os.listdir(PROJECT_ROOT)):
    print(item)

data
figures
models
notebooks


In [5]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/SEAID_Framework")

DATA_DIR = PROJECT_ROOT / "data"
FIGURES_DIR = PROJECT_ROOT / "figures"
MODELS_DIR = PROJECT_ROOT / "models"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

print("Project root:", PROJECT_ROOT.exists())
print("Data folder:", DATA_DIR.exists())
print("Figures folder:", FIGURES_DIR.exists())
print("Models folder:", MODELS_DIR.exists())
print("Notebooks folder:", NOTEBOOKS_DIR.exists())

Project root: True
Data folder: True
Figures folder: True
Models folder: True
Notebooks folder: True


In [6]:
for item in sorted(DATA_DIR.iterdir()):
    print(item.name)

processed
raw


In [7]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/SEAID_Framework")

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

FIGURES_DIR = PROJECT_ROOT / "figures"
MODELS_DIR = PROJECT_ROOT / "models"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

print("Raw data:", RAW_DATA_DIR.exists())
print("Processed data:", PROCESSED_DATA_DIR.exists())

Raw data: True
Processed data: True


In [8]:
for item in sorted(RAW_DATA_DIR.iterdir()):
    print(item.name)

assessments.csv
courses.csv
studentAssessment.csv
studentInfo.csv
studentRegistration.csv
studentVle.csv
vle.csv


In [9]:
import pandas as pd
import numpy as np

courses = pd.read_csv(RAW_DATA_DIR / "courses.csv")
student_info = pd.read_csv(RAW_DATA_DIR / "studentInfo.csv")
student_registration = pd.read_csv(
    RAW_DATA_DIR / "studentRegistration.csv"
)
student_assessment = pd.read_csv(
    RAW_DATA_DIR / "studentAssessment.csv"
)
assessments = pd.read_csv(RAW_DATA_DIR / "assessments.csv")
student_vle = pd.read_csv(RAW_DATA_DIR / "studentVle.csv")
vle = pd.read_csv(RAW_DATA_DIR / "vle.csv")

print("All OULAD datasets loaded successfully.")

All OULAD datasets loaded successfully.


In [10]:
datasets = {
    "courses": courses,
    "student_info": student_info,
    "student_registration": student_registration,
    "student_assessment": student_assessment,
    "assessments": assessments,
    "student_vle": student_vle,
    "vle": vle
}

for name, dataframe in datasets.items():
    print(f"{name}: {dataframe.shape}")

courses: (22, 3)
student_info: (32593, 12)
student_registration: (32593, 5)
student_assessment: (173912, 5)
assessments: (206, 6)
student_vle: (10655280, 6)
vle: (6364, 6)


In [11]:
student_features = student_info.copy()

print("Shape:", student_features.shape)
student_features.head()

Shape: (32593, 12)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass


In [12]:
student_keys = [
    "code_module",
    "code_presentation",
    "id_student"
]

In [13]:
duplicate_count = student_features.duplicated(
    subset=student_keys
).sum()

print("Duplicate student records:", duplicate_count)

Duplicate student records: 0


In [14]:
print("Starting columns:")

for column in student_features.columns:
    print(column)

Starting columns:
code_module
code_presentation
id_student
gender
region
highest_education
imd_band
age_band
num_of_prev_attempts
studied_credits
disability
final_result


In [15]:
student_features["final_result"].value_counts(
    dropna=False
)

,count
final_result,
Pass,12361
Withdrawn,10156
Fail,7052
Distinction,3024


In [16]:
student_features["final_result"].value_counts(
    normalize=True,
    dropna=False
).mul(100).round(2)

,proportion
final_result,
Pass,37.93
Withdrawn,31.16
Fail,21.64
Distinction,9.28


In [17]:
successful_outcomes = [
    "Pass",
    "Distinction"
]

student_features["target_success"] = (
    student_features["final_result"]
    .isin(successful_outcomes)
    .astype(int)
)

In [18]:
pd.crosstab(
    student_features["final_result"],
    student_features["target_success"]
)

target_success,0,1
final_result,,
Distinction,0,3024
Fail,7052,0
Pass,0,12361
Withdrawn,10156,0


In [19]:
target_distribution = (
    student_features["target_success"]
    .value_counts()
    .sort_index()
)

target_percentages = (
    student_features["target_success"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print("Counts:")
print(target_distribution)

print("\nPercentages:")
print(target_percentages)

Counts:
target_success
0    17208
1    15385
Name: count, dtype: int64

Percentages:
target_success
0    52.8
1    47.2
Name: proportion, dtype: float64


In [20]:
base_student_count = len(student_features)

print("Base student records:", base_student_count)

Base student records: 32593


In [21]:
print("studentVle shape:", student_vle.shape)
student_vle.head()

studentVle shape: (10655280, 6)


,code_module,code_presentation,id_student,id_site,date,sum_click
0,AAA,2013J,28400,546652,-10,4
1,AAA,2013J,28400,546652,-10,1
2,AAA,2013J,28400,546652,-10,1
3,AAA,2013J,28400,546614,-10,11
4,AAA,2013J,28400,546714,-10,1


In [22]:
print(student_vle.columns.tolist())

['code_module', 'code_presentation', 'id_student', 'id_site', 'date', 'sum_click']


In [23]:
engagement_features = (
    student_vle
    .groupby(student_keys, as_index=False)
    .agg(
        total_clicks=("sum_click", "sum"),
        average_clicks_per_record=("sum_click", "mean"),
        median_clicks_per_record=("sum_click", "median"),
        maximum_clicks_in_record=("sum_click", "max"),
        vle_records=("sum_click", "size"),
        active_days=("date", "nunique"),
        unique_vle_activities=("id_site", "nunique"),
        first_activity_day=("date", "min"),
        last_activity_day=("date", "max")
    )
)

In [24]:
print("Engagement feature shape:", engagement_features.shape)
engagement_features.head()

Engagement feature shape: (29228, 12)


,code_module,code_presentation,id_student,total_clicks,average_clicks_per_record,median_clicks_per_record,maximum_clicks_in_record,vle_records,active_days,unique_vle_activities,first_activity_day,last_activity_day
0,AAA,2013J,11391,934,4.765306,2.0,76,196,40,55,-5,253
1,AAA,2013J,28400,1435,3.337209,2.0,23,430,80,84,-10,239
2,AAA,2013J,30268,281,3.697368,2.0,23,76,12,22,-10,12
3,AAA,2013J,31604,2158,3.254902,2.0,22,663,123,82,-10,264
4,AAA,2013J,32885,1034,2.937500,2.0,22,352,70,66,-10,247


In [25]:
engagement_features["clicks_per_active_day"] = (
    engagement_features["total_clicks"]
    / engagement_features["active_days"].replace(0, np.nan)
)

engagement_features["activity_span_days"] = (
    engagement_features["last_activity_day"]
    - engagement_features["first_activity_day"]
)

engagement_features["clicks_per_vle_activity"] = (
    engagement_features["total_clicks"]
    / engagement_features["unique_vle_activities"].replace(0, np.nan)
)

In [26]:
engagement_features = engagement_features.replace(
    [np.inf, -np.inf],
    np.nan
)

In [27]:
student_features = student_features.merge(
    engagement_features,
    on=student_keys,
    how="left",
    validate="one_to_one"
)

In [28]:
print("Before merge:", base_student_count)
print("After merge:", len(student_features))

Before merge: 32593
After merge: 32593


In [29]:
engagement_zero_columns = [
    "total_clicks",
    "average_clicks_per_record",
    "median_clicks_per_record",
    "maximum_clicks_in_record",
    "vle_records",
    "active_days",
    "unique_vle_activities",
    "first_activity_day",
    "last_activity_day",
    "clicks_per_active_day",
    "activity_span_days",
    "clicks_per_vle_activity"
]

student_features[engagement_zero_columns] = (
    student_features[engagement_zero_columns]
    .fillna(0)
)

In [30]:
student_features["log_total_clicks"] = np.log1p(
    student_features["total_clicks"]
)

In [31]:
engagement_summary_columns = [
    "total_clicks",
    "active_days",
    "unique_vle_activities",
    "clicks_per_active_day",
    "activity_span_days",
    "log_total_clicks"
]

student_features[
    engagement_summary_columns
].describe().T

,count,mean,std,min,25%,50%,75%,max
total_clicks,32593.0,1215.141257,1692.604449,0.0,142.000000,602.000000,1585.000000,24139.000000
active_days,32593.0,55.475685,54.515290,0.0,11.000000,40.000000,85.000000,286.000000
unique_vle_activities,32593.0,60.150830,55.898555,0.0,19.000000,46.000000,86.000000,413.000000
clicks_per_active_day,32593.0,17.076258,12.399932,0.0,9.411765,15.000000,23.157895,221.200000
activity_span_days,32593.0,166.365078,104.166374,0.0,55.000000,224.000000,254.000000,293.000000
log_total_clicks,32593.0,5.706201,2.450439,0.0,4.962845,6.401917,7.368970,10.091625


In [32]:
student_features[
    engagement_summary_columns
].isna().sum()

,0
total_clicks,0
active_days,0
unique_vle_activities,0
clicks_per_active_day,0
activity_span_days,0
log_total_clicks,0


In [34]:
student_assessment_enriched = (
    student_assessment.merge(
        assessments,
        on="id_assessment",
        how="left",
        validate="many_to_one"
    )
)

print(student_assessment_enriched.shape)
student_assessment_enriched.head()

(173912, 10)


,id_assessment,id_student,date_submitted,is_banked,score,code_module,code_presentation,assessment_type,date,weight
0,1752,11391,18,0,78.0,AAA,2013J,TMA,19.0,10.0
1,1752,28400,22,0,70.0,AAA,2013J,TMA,19.0,10.0
2,1752,31604,17,0,72.0,AAA,2013J,TMA,19.0,10.0
3,1752,32885,26,0,69.0,AAA,2013J,TMA,19.0,10.0
4,1752,38053,19,0,79.0,AAA,2013J,TMA,19.0,10.0


In [35]:
print(student_assessment_enriched.columns.tolist())

['id_assessment', 'id_student', 'date_submitted', 'is_banked', 'score', 'code_module', 'code_presentation', 'assessment_type', 'date', 'weight']


In [36]:
assessment_features = (
    student_assessment_enriched
    .groupby(student_keys, as_index=False)
    .agg(
        assessments_completed=("score", "count"),
        average_score=("score", "mean"),
        median_score=("score", "median"),
        minimum_score=("score", "min"),
        maximum_score=("score", "max"),
        score_std=("score", "std"),
        average_weight=("weight", "mean"),
        total_weight_completed=("weight", "sum"),
        first_submission_day=("date_submitted", "min"),
        last_submission_day=("date_submitted", "max"),
        banked_assessments=("is_banked", "sum")
    )
)

In [37]:
assessment_features["assessment_span_days"] = (
    assessment_features["last_submission_day"]
    - assessment_features["first_submission_day"]
)

assessment_features["score_range"] = (
    assessment_features["maximum_score"]
    - assessment_features["minimum_score"]
)

In [38]:
weighted_scores = (
    student_assessment_enriched
    .assign(
        weighted_points=lambda df: df["score"] * df["weight"]
    )
    .groupby(student_keys, as_index=False)
    .agg(
        weighted_points=("weighted_points", "sum"),
        total_weight=("weight", "sum")
    )
)

weighted_scores["weighted_average_score"] = (
    weighted_scores["weighted_points"]
    / weighted_scores["total_weight"]
)

In [39]:
weighted_scores = weighted_scores[
    student_keys + ["weighted_average_score"]
]

weighted_scores.head()

,code_module,code_presentation,id_student,weighted_average_score
0,AAA,2013J,11391,82.4
1,AAA,2013J,28400,65.4
2,AAA,2013J,31604,76.3
3,AAA,2013J,32885,55.0
4,AAA,2013J,38053,66.9


In [40]:
assessment_features = assessment_features.merge(
    weighted_scores,
    on=student_keys,
    how="left",
    validate="one_to_one"
)

In [41]:
student_features = student_features.merge(
    assessment_features,
    on=student_keys,
    how="left",
    validate="one_to_one"
)

print(student_features.shape)

(32593, 40)


In [42]:
assessment_columns = [
    "assessments_completed",
    "average_score",
    "median_score",
    "minimum_score",
    "maximum_score",
    "score_std",
    "average_weight",
    "total_weight_completed",
    "first_submission_day",
    "last_submission_day",
    "banked_assessments",
    "assessment_span_days",
    "score_range",
    "weighted_average_score"
]

student_features[assessment_columns] = (
    student_features[assessment_columns]
    .fillna(0)
)

In [43]:
student_features["score_improvement"] = (
    student_features["maximum_score"]
    - student_features["average_score"]
)

student_features["assessment_intensity"] = (
    student_features["assessments_completed"]
    / (student_features["assessment_span_days"] + 1)
)

student_features["log_assessments_completed"] = np.log1p(
    student_features["assessments_completed"]
)

In [44]:
assessment_summary = [
    "assessments_completed",
    "average_score",
    "weighted_average_score",
    "assessment_span_days",
    "assessment_intensity",
    "score_std"
]

student_features[assessment_summary].describe().T

,count,mean,std,min,25%,50%,75%,max
assessments_completed,32593.0,5.330562,4.326081,0.0,1.000000,5.000000,9.000000,14.000000
average_score,32593.0,57.646621,32.926518,0.0,43.000000,70.571429,82.400000,100.000000
weighted_average_score,32593.0,50.142215,34.343470,0.0,0.000000,63.000000,78.160000,100.000000
assessment_span_days,32593.0,114.426196,89.940625,0.0,0.000000,138.000000,196.000000,589.000000
assessment_intensity,32593.0,0.144034,0.469604,0.0,0.025641,0.041475,0.059406,12.000000
score_std,32593.0,9.467725,8.777733,0.0,0.000000,8.616844,14.839137,70.003571


In [45]:
print("studentRegistration shape:", student_registration.shape)
print(student_registration.columns.tolist())

student_registration.head()

studentRegistration shape: (32593, 5)
['code_module', 'code_presentation', 'id_student', 'date_registration', 'date_unregistration']


,code_module,code_presentation,id_student,date_registration,date_unregistration
0,AAA,2013J,11391,-159.0,NaN
1,AAA,2013J,28400,-53.0,NaN
2,AAA,2013J,30268,-92.0,12.0
3,AAA,2013J,31604,-52.0,NaN
4,AAA,2013J,32885,-176.0,NaN


In [46]:
registration_features = student_registration.copy()

registration_features["registered_before_start"] = (
    registration_features["date_registration"] < 0
).astype(int)

registration_features["registered_after_start"] = (
    registration_features["date_registration"] > 0
).astype(int)

registration_features["days_registered_before_start"] = (
    -registration_features["date_registration"]
).clip(lower=0)

registration_features["withdrew"] = (
    registration_features["date_unregistration"].notna()
).astype(int)

In [47]:
registration_features["days_until_withdrawal"] = (
    registration_features["date_unregistration"]
    - registration_features["date_registration"]
)

registration_features["early_withdrawal"] = (
    registration_features["date_unregistration"].notna()
    & (registration_features["date_unregistration"] <= 60)
).astype(int)

In [48]:
registration_features["days_until_withdrawal"] = (
    registration_features["days_until_withdrawal"]
    .fillna(0)
)

In [49]:
registration_features["registration_timing"] = pd.cut(
    registration_features["date_registration"],
    bins=[
        -np.inf,
        -60,
        -30,
        -1,
        0,
        30,
        np.inf
    ],
    labels=[
        "very_early",
        "early",
        "moderately_early",
        "course_start",
        "late",
        "very_late"
    ]
)

In [50]:
registration_features["registration_timing"].value_counts(
    dropna=False
)

,count
registration_timing,
very_early,15482
early,8625
moderately_early,8205
late,218
NaN,45
very_late,16
course_start,2


In [51]:
registration_columns = student_keys + [
    "date_registration",
    "date_unregistration",
    "registered_before_start",
    "registered_after_start",
    "days_registered_before_start",
    "withdrew",
    "days_until_withdrawal",
    "early_withdrawal",
    "registration_timing"
]

registration_features = registration_features[
    registration_columns
]

In [52]:
registration_features.duplicated(
    subset=student_keys
).sum()

np.int64(0)

In [53]:
student_features = student_features.merge(
    registration_features,
    on=student_keys,
    how="left",
    validate="one_to_one"
)

print("Before merge:", base_student_count)
print("After merge:", len(student_features))

Before merge: 32593
After merge: 32593


In [54]:
registration_numeric_columns = [
    "date_registration",
    "date_unregistration",
    "registered_before_start",
    "registered_after_start",
    "days_registered_before_start",
    "withdrew",
    "days_until_withdrawal",
    "early_withdrawal"
]

student_features[registration_numeric_columns] = (
    student_features[registration_numeric_columns]
    .fillna(0)
)

student_features["registration_timing"] = (
    student_features["registration_timing"]
    .astype("object")
    .fillna("unknown")
)

In [55]:
student_features[
    registration_numeric_columns
].describe().T

,count,mean,std,min,25%,50%,75%,max
date_registration,32593.0,-69.315467,49.293930,-322.0,-100.0,-57.0,-29.0,167.0
date_unregistration,32593.0,15.376277,51.281833,-365.0,0.0,0.0,0.0,444.0
registered_before_start,32593.0,0.991379,0.092452,0.0,1.0,1.0,1.0,1.0
registered_after_start,32593.0,0.007179,0.084428,0.0,0.0,0.0,0.0,1.0
days_registered_before_start,32593.0,69.403952,49.129177,0.0,29.0,57.0,100.0,322.0
withdrew,32593.0,0.309023,0.462098,0.0,0.0,0.0,1.0,1.0
days_until_withdrawal,32593.0,39.647348,76.125435,0.0,0.0,0.0,45.0,531.0
early_withdrawal,32593.0,0.191207,0.393257,0.0,0.0,0.0,0.0,1.0


In [56]:
student_features[
    registration_numeric_columns
    + ["registration_timing"]
].isna().sum()

,0
date_registration,0
date_unregistration,0
registered_before_start,0
registered_after_start,0
days_registered_before_start,0
withdrew,0
days_until_withdrawal,0
early_withdrawal,0
registration_timing,0


In [57]:
demographic_columns = [
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "disability",
    "num_of_prev_attempts",
    "studied_credits"
]

student_features[demographic_columns].head()

,gender,region,highest_education,imd_band,age_band,disability,num_of_prev_attempts,studied_credits
0,M,East Anglian Region,HE Qualification,90-100%,55<=,N,0,240
1,F,Scotland,HE Qualification,20-30%,35-55,N,0,60
2,F,North Western Region,A Level or Equivalent,30-40%,35-55,Y,0,60
3,F,South East Region,A Level or Equivalent,50-60%,35-55,N,0,60
4,F,West Midlands Region,Lower Than A Level,50-60%,0-35,N,0,60


In [58]:
student_features[demographic_columns].isna().sum()

,0
gender,0
region,0
highest_education,0
imd_band,1111
age_band,0
disability,0
num_of_prev_attempts,0
studied_credits,0


In [59]:
student_features["imd_band"] = (
    student_features["imd_band"]
    .fillna("Unknown")
)

student_features["region"] = (
    student_features["region"]
    .fillna("Unknown")
)

student_features["highest_education"] = (
    student_features["highest_education"]
    .fillna("Unknown")
)

student_features["age_band"] = (
    student_features["age_band"]
    .fillna("Unknown")
)

student_features["gender"] = (
    student_features["gender"]
    .fillna("Unknown")
)

student_features["disability"] = (
    student_features["disability"]
    .fillna("Unknown")
)

In [60]:
student_features["gender"] = (
    student_features["gender"]
    .map({
        "M": 0,
        "F": 1
    })
)

student_features["disability"] = (
    student_features["disability"]
    .map({
        "N": 0,
        "Y": 1
    })
)

In [61]:
categorical_columns = [
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "registration_timing"
]

student_features = pd.get_dummies(
    student_features,
    columns=categorical_columns,
    drop_first=True,
    dtype=int
)

In [62]:
print(student_features.shape)

student_features.head()

(32593, 81)


,code_module,code_presentation,id_student,gender,num_of_prev_attempts,studied_credits,disability,final_result,target_success,total_clicks,...,imd_band_90-100%,imd_band_Unknown,age_band_35-55,age_band_55<=,registration_timing_early,registration_timing_late,registration_timing_moderately_early,registration_timing_unknown,registration_timing_very_early,registration_timing_very_late
0,AAA,2013J,11391,0,0,240,0,Pass,1,934.0,...,1,0,0,1,0,0,0,0,1,0
1,AAA,2013J,28400,1,0,60,0,Pass,1,1435.0,...,0,0,1,0,1,0,0,0,0,0
2,AAA,2013J,30268,1,0,60,1,Withdrawn,0,281.0,...,0,0,1,0,0,0,0,0,1,0
3,AAA,2013J,31604,1,0,60,0,Pass,1,2158.0,...,0,0,1,0,1,0,0,0,0,0
4,AAA,2013J,32885,1,0,60,0,Pass,1,1034.0,...,0,0,0,0,0,0,0,0,1,0


In [63]:
missing = (
    student_features
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing[missing > 0]

,0


In [64]:
print("Rows:", student_features.shape[0])
print("Columns:", student_features.shape[1])

student_features.info()

Rows: 32593
Columns: 81
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 81 columns):
 #   Column                                         Non-Null Count  Dtype  
---  ------                                         --------------  -----  
 0   code_module                                    32593 non-null  object 
 1   code_presentation                              32593 non-null  object 
 2   id_student                                     32593 non-null  int64  
 3   gender                                         32593 non-null  int64  
 4   num_of_prev_attempts                           32593 non-null  int64  
 5   studied_credits                                32593 non-null  int64  
 6   disability                                     32593 non-null  int64  
 7   final_result                                   32593 non-null  object 
 8   target_success                                 32593 non-null  int64  
 9   total_clicks              

In [65]:
output_path = (
    PROCESSED_DATA_DIR
    / "final_modeling_dataset.csv"
)

student_features.to_csv(
    output_path,
    index=False
)

print("Saved to:")
print(output_path)

Saved to:
/content/drive/MyDrive/SEAID_Framework/data/processed/final_modeling_dataset.csv


In [66]:
loaded = pd.read_csv(output_path)

print(loaded.shape)

loaded.head()

(32593, 81)


,code_module,code_presentation,id_student,gender,num_of_prev_attempts,studied_credits,disability,final_result,target_success,total_clicks,...,imd_band_90-100%,imd_band_Unknown,age_band_35-55,age_band_55<=,registration_timing_early,registration_timing_late,registration_timing_moderately_early,registration_timing_unknown,registration_timing_very_early,registration_timing_very_late
0,AAA,2013J,11391,0,0,240,0,Pass,1,934.0,...,1,0,0,1,0,0,0,0,1,0
1,AAA,2013J,28400,1,0,60,0,Pass,1,1435.0,...,0,0,1,0,1,0,0,0,0,0
2,AAA,2013J,30268,1,0,60,1,Withdrawn,0,281.0,...,0,0,1,0,0,0,0,0,1,0
3,AAA,2013J,31604,1,0,60,0,Pass,1,2158.0,...,0,0,1,0,1,0,0,0,0,0
4,AAA,2013J,32885,1,0,60,0,Pass,1,1034.0,...,0,0,0,0,0,0,0,0,1,0


In [67]:
student_features = pd.read_csv(
    DATA_DIR / "processed" / "final_modeling_dataset.csv"
)

| Feature                 | Description                                   |
| ----------------------- | --------------------------------------------- |
| total_clicks            | Total VLE clicks                              |
| active_days             | Number of unique days active                  |
| weighted_average_score  | Assessment score weighted by assessment value |
| assessment_span_days    | Days between first and last submission        |
| registered_before_start | Registered before course start                |
| target_success          | Pass/Distinction = 1; Fail/Withdrawn = 0      |


In [68]:
print(student_features.shape)

student_features.isna().sum().sum()

student_features.duplicated(
    subset=student_keys
).sum()

(32593, 81)


np.int64(0)

In [70]:
feature_dictionary = pd.DataFrame({
    "Feature": student_features.columns,
    "Data Type": student_features.dtypes.astype(str),
    "Missing Values": student_features.isna().sum().values,
    "Percent Missing": (
        student_features.isna().mean().values * 100
    ).round(2)
})

feature_dictionary = feature_dictionary.sort_values(
    by="Feature"
).reset_index(drop=True)

feature_dictionary.to_csv(
    PROCESSED_DATA_DIR / "feature_dictionary.csv",
    index=False
)

print("Feature dictionary saved to:")
print(PROCESSED_DATA_DIR / "feature_dictionary.csv")
print(f"\nTotal features: {len(feature_dictionary)}")

feature_dictionary.head()

Feature dictionary saved to:
/content/drive/MyDrive/SEAID_Framework/data/processed/feature_dictionary.csv

Total features: 81


,Feature,Data Type,Missing Values,Percent Missing
0,active_days,float64,0,0.0
1,activity_span_days,float64,0,0.0
2,age_band_35-55,int64,0,0.0
3,age_band_55<=,int64,0,0.0
4,assessment_intensity,float64,0,0.0
